# Compare recent InstanSeg checkpoints on one fixed CPDMI validation set

This notebook compares the recent multihead InstanSeg training outputs under the shared
/data1/lowes/ratnayn/Data/instanseg root. It is designed for the CPDMI question:

> How do the CPDMI-only and CPDMI+TissueNet (mixed) models perform when evaluated on exactly the same CPDMI validation records?

The notebook:

1. discovers recent multihead model directories and records their training provenance;
2. loads the saved segmentation_dataset.pth and selects CPDMI records from its Validation split;
3. freezes that selection in a manifest containing validation indices, filenames, shapes, dtypes, and array hashes;
4. applies deterministic percentile normalization and physical-pixel-size rescaling, with no random crop or flip;
5. evaluates every selected checkpoint with one shared postprocessing configuration;
6. evaluates the public fluorescence_nuclei_and_cells v0.1.1 network through its supported inference wrapper; and
7. writes compact CSV/JSON analysis artifacts and displays threshold curves, per-record scores, and optional overlays.

The default checkpoint is model_weights.pth, which is the trainer's best-validation checkpoint at the time
the model directory was last written. Change CHECKPOINT_FILENAME to latest_checkpoint.pth when you
want the current last-epoch weights instead. The auto-discovery includes mixed models and _l40s_fallback
directories as separate stochastic runs. Legacy shared-decoder directories are excluded by the default
*_multihead* pattern. The public baseline is downloaded lazily only when RUN_EVALUATION=True.

In [ ]:
import ast
from contextlib import contextmanager
import gc
import hashlib
import json
import math
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

TRAINING_ROOT = Path(
    os.environ.get('INSTANSEG_TRAINING_ROOT', '/data1/lowes/ratnayn/Data/instanseg')
).expanduser().resolve()
DATASET_DIR = TRAINING_ROOT / 'datasets'
DATASET_FILE = DATASET_DIR / 'segmentation_dataset.pth'
MODEL_ROOT = TRAINING_ROOT / 'models'
RUN_ROOT = TRAINING_ROOT / 'slurm_runs'
RESULTS_ROOT = Path(
    os.environ.get(
        'INSTANSEG_COMPARISON_ROOT',
        str(TRAINING_ROOT / 'model_comparisons' / 'cpdmi_validation'),
    )
).expanduser().resolve()
SOURCE_ROOT = Path(
    os.environ.get(
        'INSTANSEG_EVAL_SOURCE_ROOT',
        '/data1/lowes/ratnayn/Data/instanseg/slurm_runs/'
        'instanseg_multihead_0325_20260825/source/instanseg',
    )
).expanduser().resolve()

PUBLIC_MODEL_NAME = 'fluorescence_nuclei_and_cells'
PUBLIC_MODEL_VERSION = '0.1.1'
PUBLIC_MODEL_LABEL = f'public_{PUBLIC_MODEL_NAME}_v{PUBLIC_MODEL_VERSION}'
PUBLIC_MODEL_CACHE_ROOT = Path(
    os.environ.get(
        'INSTANSEG_PUBLIC_MODEL_CACHE',
        str(TRAINING_ROOT / 'public_model_cache'),
    )
).expanduser().resolve()
INCLUDE_PUBLIC_MODEL = True
PUBLIC_MODEL_TILE_SIZE = int(os.environ.get('INSTANSEG_PUBLIC_MODEL_TILE_SIZE', '512'))
# Leave the public baseline on the same device as the trained models by default. The
# public model's raw network is safe to run there; its fragile TorchScript postprocessor
# is replaced below by a Python implementation with a coalesced, symmetric IoU adjacency.
PUBLIC_INFERENCE_DEVICE_OVERRIDE = os.environ.get('INSTANSEG_PUBLIC_INFERENCE_DEVICE')

SPLIT = 'Validation'
DATASET_NAME = 'CPDMI_2023'
EXPECTED_CPDMI_VALIDATION_COUNT = 30

# None discovers every matching output with a usable checkpoint. Set an explicit list to
# compare only one run per configuration or to include a non-multihead directory.
MODEL_NAMES = None
MODEL_GLOB = '*_multihead*'
EXCLUDE_MODEL_SUBSTRINGS = ('_smoke',)

# model_weights.pth is the best checkpoint saved by the trainer. The loader below also
# supports latest_checkpoint.pth and best_model_weights.pth.
CHECKPOINT_FILENAME = 'model_weights.pth'

# Full evaluation is deliberately gated because it requires a CUDA allocation and can take
# several minutes per model. The manifest is still created when this is False.
RUN_EVALUATION = True
SAVE_ARTIFACTS = True
INFERENCE_BATCH_SIZE = 1
FIXED_TILE_SIZE = None  # None uses each trained tile_size or the public baseline's 512-pixel default.
INFERENCE_DIMENSION_MULTIPLE = 16  # keep encoder/decoder dimensions even at small image sizes.
GALLERY_COUNT = 2

# These are the training-time postprocessing defaults, made explicit and shared across all
# checkpoints. Native cell/nucleus reconciliation is handled by the shared tile stitcher.
POSTPROCESSING = {
    'mask_threshold': 0.53,
    'peak_distance': 4,
    'seed_threshold': 0.5,
    'overlap_threshold': 0.5,
    'mean_threshold': -10000.0,
    'window_size': 128,
    'min_size': 10,
    'cleanup_fragments': False,
    'max_seeds': 2000,
    'resolve_cell_and_nucleus': True,
}
THRESHOLDS = [round(float(x), 2) for x in np.linspace(0.5, 1.0, 10)]

print('Training root:', TRAINING_ROOT)
print('Model root:', MODEL_ROOT)
print('Dataset:', DATASET_FILE)
print('Run provenance root:', RUN_ROOT)
print('Results root:', RESULTS_ROOT)
print('Public model:', PUBLIC_MODEL_LABEL)
print('Public model cache:', PUBLIC_MODEL_CACHE_ROOT)
print('Public inference device override:', PUBLIC_INFERENCE_DEVICE_OVERRIDE or 'auto')
print('Checkpoint:', CHECKPOINT_FILENAME)
print('Evaluation enabled:', RUN_EVALUATION)

## Environment and source snapshot

The model directories were produced by the fork snapshot recorded in the multihead SLURM batch.
Import that same snapshot before importing InstanSeg so the checkpoint architecture and the tile
stitching behavior are explicit. If the source path changes, set INSTANSEG_EVAL_SOURCE_ROOT and
restart the kernel before running this cell. The comparison can be smoke-tested on CPU, but the
full 30-record evaluation is intended for the instanseg_training CUDA environment. The public baseline
is downloaded into PUBLIC_MODEL_CACHE_ROOT only when the evaluation cell is enabled.

In [ ]:
MPLCONFIGDIR = Path('/tmp/mif_instanseg_compare_matplotlib')
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))
os.environ['INSTANSEG_DATASET_PATH'] = str(DATASET_DIR)

if not (SOURCE_ROOT / 'instanseg').is_dir():
    raise FileNotFoundError(
        f'Expected an InstanSeg source tree at {SOURCE_ROOT / "instanseg"}. '
        'Set INSTANSEG_EVAL_SOURCE_ROOT to the immutable training snapshot.'
    )
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

import torch
import instanseg
from instanseg import InstanSeg as PublicInstanSeg

from instanseg.utils.augmentations import Augmentations
from instanseg.utils.data_loader import get_image
from instanseg.utils.loss import instanseg_loss as loss_utils
from instanseg.utils.loss.instanseg_loss import InstanSeg as LossInstanSeg
from instanseg.utils.metrics import matching_torch
from instanseg.utils.model_loader import (
    build_model_from_dict,
    has_adaptor_net_state_dict,
    has_pixel_classifier_model,
    has_pixel_classifier_state_dict,
    read_model_args_from_csv,
    remove_module_prefix_from_dict,
)
from instanseg.utils.utils import _filter_kwargs
from instanseg.utils.models.ChannelInvariantNet import AdaptorNetWrapper, has_AdaptorNet
from instanseg.utils.tiling import (
    _instanseg_padding,
    _recover_padding,
    _sliding_window_inference,
)

instanseg_import_path = Path(instanseg.__file__).resolve()
if not instanseg_import_path.is_relative_to(SOURCE_ROOT / 'instanseg'):
    raise RuntimeError(
        f'InstanSeg imported from {instanseg_import_path}, not from the requested '
        f'snapshot {SOURCE_ROOT}. Restart the kernel after fixing sys.path.'
    )

if not DATASET_FILE.is_file():
    raise FileNotFoundError(DATASET_FILE)
if not MODEL_ROOT.is_dir():
    raise FileNotFoundError(MODEL_ROOT)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
PUBLIC_INFERENCE_DEVICE = PUBLIC_INFERENCE_DEVICE_OVERRIDE or DEVICE
print('InstanSeg source:', instanseg_import_path)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', DEVICE)
print('Public inference device:', PUBLIC_INFERENCE_DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(torch.cuda.current_device()))
elif RUN_EVALUATION:
    print('CUDA is unavailable; evaluation will run on CPU and may be slow.')

## Discover and audit candidate checkpoints

Auto-discovery is restricted to recent multihead output names. Mixed CPDMI+TissueNet models are expected
to appear in the table, and any retained L40S fallback is kept as a separate run because it has a
different stochastic training trajectory. Training-time F1 is shown only as provenance; it is not used
as the comparison result because the mixed models were selected with a pooled CPDMI+TissueNet validation
stream rather than this fixed CPDMI-only set.

In [ ]:
def _safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')


def _as_source_dataset_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [str(item) for item in value]
    try:
        parsed = ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        parsed = value
    if isinstance(parsed, (list, tuple)):
        return [str(item) for item in parsed]
    return [str(parsed)]


def _run_batches_for_model(model_name):
    batches = []
    if not RUN_ROOT.is_dir():
        return batches
    for matrix_path in sorted(RUN_ROOT.glob('*/experiment_matrix.json')):
        try:
            matrix = json.loads(matrix_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if any(exp.get('name') == model_name for exp in matrix.get('experiments', [])):
            batches.append(matrix_path.parent.name)
    return batches


def _checkpoint_metadata(checkpoint_path):
    checkpoint = _safe_torch_load(checkpoint_path)
    training_config = checkpoint.get('training_config') or {}
    metadata = {
        'checkpoint_epoch': checkpoint.get('epoch'),
        'checkpoint_phase': checkpoint.get('phase'),
        'training_best_f1': checkpoint.get('best_f1_score'),
        'checkpoint_schema_version': checkpoint.get('checkpoint_schema_version'),
        'checkpoint_training_config': training_config,
    }
    del checkpoint
    return metadata


if MODEL_NAMES is None:
    candidate_dirs = [
        path for path in sorted(MODEL_ROOT.glob(MODEL_GLOB))
        if path.is_dir()
        and not any(token in path.name for token in EXCLUDE_MODEL_SUBSTRINGS)
    ]
    SELECTED_MODEL_NAMES = [path.name for path in candidate_dirs]
else:
    SELECTED_MODEL_NAMES = list(MODEL_NAMES)

if not SELECTED_MODEL_NAMES:
    raise RuntimeError(
        f'No model directories matched {MODEL_GLOB!r} under {MODEL_ROOT}. '
        'Set MODEL_NAMES explicitly.'
    )

MODEL_ROWS = []
MODEL_CONFIGS = {}
for model_name in SELECTED_MODEL_NAMES:
    model_dir = MODEL_ROOT / model_name
    if not model_dir.is_dir():
        raise FileNotFoundError(model_dir)
    log_path = model_dir / 'experiment_log.csv'
    checkpoint_path = model_dir / CHECKPOINT_FILENAME
    if not log_path.is_file():
        raise FileNotFoundError(log_path)
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)

    config = read_model_args_from_csv(path=MODEL_ROOT, folder=model_name)
    checkpoint_info = _checkpoint_metadata(checkpoint_path)
    MODEL_CONFIGS[model_name] = config

    requested_pixel_size = config.get('requested_pixel_size', config.get('pixel_size'))
    MODEL_ROWS.append({
        'model': model_name,
        'source_dataset': '+'.join(_as_source_dataset_list(config.get('source_dataset'))),
        'requested_pixel_size_um': float(requested_pixel_size),
        'tile_size_px': int(config.get('tile_size')),
        'window_size_px': int(config.get('window_size')),
        'multihead': bool(config.get('multihead', False)),
        'checkpoint': CHECKPOINT_FILENAME,
        'checkpoint_epoch': checkpoint_info['checkpoint_epoch'],
        'checkpoint_phase': checkpoint_info['checkpoint_phase'],
        'training_best_f1': checkpoint_info['training_best_f1'],
        'complete': (model_dir / 'training_complete.json').is_file(),
        'slurm_batches': ', '.join(_run_batches_for_model(model_name)),
    })

MODEL_TABLE = pd.DataFrame(MODEL_ROWS)
if MODEL_NAMES is None and not any('tissuenet' in name.lower() for name in SELECTED_MODEL_NAMES):
    raise RuntimeError('Auto-discovery found no mixed CPDMI+TissueNet model.')
if not MODEL_TABLE['multihead'].all():
    raise ValueError('The default comparison expects multihead checkpoints.')

display(MODEL_TABLE)
print(f'Selected {len(SELECTED_MODEL_NAMES)} model directories.')
EVALUATION_MODEL_NAMES = list(SELECTED_MODEL_NAMES)
if INCLUDE_PUBLIC_MODEL:
    EVALUATION_MODEL_NAMES.append(PUBLIC_MODEL_LABEL)

display(
    pd.DataFrame([{
        'model': PUBLIC_MODEL_LABEL,
        'source_dataset': 'public InstanSeg release',
        'requested_pixel_size_um': 'loaded from model',
        'tile_size_px': PUBLIC_MODEL_TILE_SIZE,
        'checkpoint': 'fluorescence_nuclei_and_cells',
        'version': PUBLIC_MODEL_VERSION,
    }])
)
print(f'Evaluation models including public baseline: {len(EVALUATION_MODEL_NAMES)}')
print(
    'Mixed candidates:',
    [name for name in SELECTED_MODEL_NAMES if 'tissuenet' in name.lower()],
)

## Freeze the CPDMI validation manifest

The saved combined dataset contains 30 CPDMI records in the Validation split. The manifest uses the
original list index plus the filename and cryptographic hashes of the raw image and available labels.
On later runs, a changed order, filename, shape, dtype, or array value raises an error instead of
silently comparing a different validation set. Cell-only CPDMI records remain in the set; their
nucleus compartment is marked unavailable and is excluded only for the nuclear metric.

In [ ]:
def _resolve_array(value):
    if isinstance(value, (str, Path)):
        return np.asarray(get_image(str(value)))
    return np.asarray(value)


def _array_hash(value):
    if value is None:
        return None
    array = np.ascontiguousarray(_resolve_array(value))
    return hashlib.sha256(array.tobytes(order='C')).hexdigest()


def _nucleus_value(item):
    if item.get('nucleus_masks') is not None:
        return item.get('nucleus_masks')
    return item.get('masks')


def _cell_value(item):
    return item.get('cell_masks')


def _labels_from_item(item):
    nucleus = _nucleus_value(item)
    cell = _cell_value(item)
    if nucleus is None and cell is None:
        raise ValueError(f'No labels found for {item.get("filename", "<unnamed>")}.')
    reference = _resolve_array(nucleus if nucleus is not None else cell)
    if nucleus is None:
        nucleus = np.full(reference.shape, -1, dtype=np.int32)
    else:
        nucleus = _resolve_array(nucleus).astype(np.int32, copy=False)
    if cell is None:
        cell = np.full(reference.shape, -1, dtype=np.int32)
    else:
        cell = _resolve_array(cell).astype(np.int32, copy=False)
    if nucleus.shape != cell.shape:
        raise ValueError(
            f'Label shape mismatch for {item.get("filename", "<unnamed>")}: '
            f'{nucleus.shape} versus {cell.shape}'
        )
    return np.stack((nucleus, cell), axis=0)


def _manifest_record(source_index, item):
    image = _resolve_array(item['image'])
    nucleus = _nucleus_value(item)
    cell = _cell_value(item)
    return {
        'record_id': (
            f'{SPLIT}[{source_index}]::{DATASET_NAME}::'
            f'{item.get("filename", source_index)}'
        ),
        'source_index': int(source_index),
        'filename': str(item.get('filename', source_index)),
        'parent_dataset': str(item.get('parent_dataset')),
        'platform': str(item.get('platform', '')),
        'pixel_size_um': float(item['pixel_size']),
        'image_shape': [int(value) for value in image.shape],
        'image_dtype': str(image.dtype),
        'image_sha256': _array_hash(item['image']),
        'nucleus_annotation': nucleus is not None,
        'nucleus_shape': (
            [int(value) for value in _resolve_array(nucleus).shape]
            if nucleus is not None else None
        ),
        'nucleus_dtype': (
            str(_resolve_array(nucleus).dtype) if nucleus is not None else None
        ),
        'nucleus_sha256': _array_hash(nucleus),
        'cell_annotation': cell is not None,
        'cell_shape': (
            [int(value) for value in _resolve_array(cell).shape]
            if cell is not None else None
        ),
        'cell_dtype': str(_resolve_array(cell).dtype) if cell is not None else None,
        'cell_sha256': _array_hash(cell),
    }


combined_dataset = _safe_torch_load(DATASET_FILE)
if SPLIT not in combined_dataset:
    raise KeyError(f'Missing {SPLIT!r} split in {DATASET_FILE}.')
validation_all = combined_dataset[SPLIT]

candidate_indices = [
    index for index, item in enumerate(validation_all)
    if item.get('parent_dataset') == DATASET_NAME
    and not item.get('duplicate', False)
    and (_nucleus_value(item) is not None or _cell_value(item) is not None)
]
if len(candidate_indices) != EXPECTED_CPDMI_VALIDATION_COUNT:
    raise RuntimeError(
        f'Expected {EXPECTED_CPDMI_VALIDATION_COUNT} CPDMI validation records, '
        f'found {len(candidate_indices)}. Inspect the dataset before proceeding.'
    )

current_manifest_records = [
    _manifest_record(index, validation_all[index])
    for index in candidate_indices
]
manifest_payload = {
    'manifest_schema_version': 1,
    'dataset_file': str(DATASET_FILE),
    'split': SPLIT,
    'filter': {
        'parent_dataset': DATASET_NAME,
        'exclude_duplicate': True,
        'target_compartments': ['nuclei', 'cells'],
    },
    'records': current_manifest_records,
}

MANIFEST_PATH = RESULTS_ROOT / 'cpdmi_validation_manifest.json'
if MANIFEST_PATH.is_file():
    stored_manifest = json.loads(MANIFEST_PATH.read_text())
    if stored_manifest.get('split') != SPLIT or stored_manifest.get('filter') != manifest_payload['filter']:
        raise RuntimeError(f'Existing manifest has incompatible selection settings: {MANIFEST_PATH}')
    if stored_manifest.get('records') != current_manifest_records:
        raise RuntimeError(
            f'Existing manifest does not match the current saved dataset: {MANIFEST_PATH}. '
            'Use a new RESULTS_ROOT only after deliberately reviewing the dataset change.'
        )
    manifest_payload = stored_manifest
    print('Reused verified manifest:', MANIFEST_PATH)
elif SAVE_ARTIFACTS:
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(manifest_payload, indent=2) + '\n')
    print('Wrote new manifest:', MANIFEST_PATH)
else:
    print('Manifest is in memory only; set SAVE_ARTIFACTS=True to persist it.')

MANIFEST_RECORDS = manifest_payload['records']
VALIDATION_RECORDS = [
    validation_all[int(record['source_index'])] for record in MANIFEST_RECORDS
]
RECORD_LOOKUP = {
    record['record_id']: (int(record['source_index']), validation_all[int(record['source_index'])])
    for record in MANIFEST_RECORDS
}
MANIFEST_TABLE = pd.DataFrame(MANIFEST_RECORDS)[[
    'record_id', 'source_index', 'filename', 'platform', 'pixel_size_um',
    'image_shape', 'nucleus_annotation', 'cell_annotation',
]]
display(MANIFEST_TABLE)
print('Fixed CPDMI validation records:', len(MANIFEST_RECORDS))

del combined_dataset
del validation_all

## Deterministic validation preprocessing

The trainer's test transform normalizes and rescales each record, but it also applies a random flip and
samples random crops. For a fair whole-record comparison, this notebook keeps the same percentile
normalization and physical rescaling while setting crop=False and omitting the random flip. Each model
still receives its own requested pixel size and trained tile size; this preserves the intended 0.5 versus
0.325 resolution and 256 versus 384 tile comparisons on the same biological records.

In [ ]:
VALIDATION_AUGMENTER = Augmentations()
PREPARED_CACHE = {}


def prepare_validation_record(item, requested_pixel_size):
    image = _resolve_array(item['image'])
    labels = _labels_from_item(item)

    image_tensor, labels_tensor = VALIDATION_AUGMENTER.to_tensor(
        image,
        labels,
        normalize=False,
    )
    image_tensor, _ = VALIDATION_AUGMENTER.normalize(image_tensor)

    native_pixel_size = float(item['pixel_size'])
    image_tensor, labels_tensor = VALIDATION_AUGMENTER.torch_rescale(
        image_tensor,
        labels_tensor,
        current_pixel_size=native_pixel_size,
        requested_pixel_size=float(requested_pixel_size),
        crop=False,
        modality='Fluorescence',
    )
    image_tensor = image_tensor.contiguous().float()
    labels_tensor = labels_tensor.contiguous().to(torch.int32)
    if image_tensor.shape[-2:] != labels_tensor.shape[-2:]:
        raise ValueError(
            f'Prepared image/label shape mismatch for {item.get("filename", "<unnamed>")}: '
            f'{tuple(image_tensor.shape)} versus {tuple(labels_tensor.shape)}'
        )
    return image_tensor, labels_tensor


def get_prepared_record(source_index, item, requested_pixel_size):
    cache_key = (int(source_index), round(float(requested_pixel_size), 6))
    if cache_key not in PREPARED_CACHE:
        PREPARED_CACHE[cache_key] = prepare_validation_record(
            item,
            requested_pixel_size,
        )
    return PREPARED_CACHE[cache_key]


first_config = MODEL_CONFIGS[SELECTED_MODEL_NAMES[0]]
first_index, first_item = RECORD_LOOKUP[MANIFEST_RECORDS[0]['record_id']]
sample_image, sample_labels = get_prepared_record(
    first_index,
    first_item,
    first_config['requested_pixel_size'],
)
print('Example prepared image:', tuple(sample_image.shape), sample_image.dtype)
print('Example prepared labels:', tuple(sample_labels.shape), sample_labels.dtype)

## Load a checkpoint and build a common tiled predictor

The saved checkpoint contains the raw two-decoder network, its pixel classifier, and the channel-invariant
adaptor. The loader below reconstructs those components from the experiment log and then loads any selected
checkpoint filename. Predictions use the same InstanSeg postprocessor and the shared medium-mode stitcher
for every trained checkpoint; the explicit postprocessing dictionary prevents newer defaults from silently
changing the comparison. The public fluorescence_nuclei_and_cells v0.1.1 network is loaded through the
supported public wrapper and evaluated on the same prepared records; its raw TorchScript network is
run on the configured inference device, while its fragile TorchScript postprocessing is replaced by a
Python equivalent. The replacement coalesces and explicitly symmetrizes the IoU adjacency before using
the public connected-components routine, avoiding both the CUDA symmetry exception and the CPU sparse
allocator crash. Inputs are padded to a model-safe spatial multiple before tiling and predictions are
cropped back to the original shape.

In [ ]:
from torch import nn


def _optional_float(value):
    if isinstance(value, (float, np.floating)) and np.isnan(value):
        return None
    return value


def load_checkpoint_for_evaluation(model_name):
    config = read_model_args_from_csv(path=MODEL_ROOT, folder=model_name)
    model_dir = MODEL_ROOT / model_name
    checkpoint_path = model_dir / CHECKPOINT_FILENAME
    checkpoint = _safe_torch_load(checkpoint_path)
    state = remove_module_prefix_from_dict(checkpoint['model_state_dict'])

    method = LossInstanSeg(
        n_sigma=int(config['n_sigma']),
        binary_loss_fn_str=str(config['binary_loss_fn']),
        seed_loss_fn=str(config['seed_loss_fn']),
        device=DEVICE,
        cells_and_nuclei=bool(config['cells_and_nuclei']),
        window_size=int(config['window_size']),
        dim_coords=int(config['dim_coords']),
        dim_seeds=int(config['dim_seeds']),
        feature_engineering_function=str(config['feature_engineering']),
        bg_weight=_optional_float(config.get('bg_weight')),
    )
    model = build_model_from_dict(config, random_seed=None)

    if has_pixel_classifier_state_dict(state) and not has_pixel_classifier_model(model):
        model = method.initialize_pixel_classifier(
            model,
            MLP_width=int(config.get('mlp_width', 5)),
        )
    if has_adaptor_net_state_dict(state) and not has_AdaptorNet(model):
        model = AdaptorNetWrapper(
            model,
            norm=config.get('norm'),
            adaptor_net_str=str(config.get('adaptor_net_str', '1')),
        )

    model.load_state_dict(state, strict=True)
    del state
    del checkpoint
    model = model.to(DEVICE)
    model.eval()

    if not hasattr(method, 'pixel_classifier'):
        raise RuntimeError(f'Pixel classifier was not reconstructed for {model_name}.')
    return model, config, method


def load_public_model_for_evaluation():
    model_index_path = SOURCE_ROOT / 'instanseg' / 'bioimageio_models' / 'model-index.json'
    model_index = json.loads(model_index_path.read_text())
    public_entries = [
        entry for entry in model_index
        if entry.get('name') == PUBLIC_MODEL_NAME
    ]
    if not public_entries:
        raise RuntimeError(f'Public model is absent from {model_index_path}.')
    if public_entries[0].get('version') != PUBLIC_MODEL_VERSION:
        raise RuntimeError(
            f'Expected public model version {PUBLIC_MODEL_VERSION}, found '
            f'{public_entries[0].get("version")} in {model_index_path}.'
        )

    PUBLIC_MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['INSTANSEG_BIOIMAGEIO_PATH'] = str(PUBLIC_MODEL_CACHE_ROOT)
    public_runner = PublicInstanSeg(
        model_type=PUBLIC_MODEL_NAME,
        device=PUBLIC_INFERENCE_DEVICE,
        verbosity=0,
        channels_last=False,
    )
    public_network = public_runner.instanseg
    public_network.eval()
    public_pixel_size = float(public_network.pixel_size)
    if not np.isfinite(public_pixel_size) or public_pixel_size <= 0:
        raise RuntimeError(f'Public model has invalid pixel size: {public_pixel_size}.')
    public_config = {
        'requested_pixel_size': public_pixel_size,
        'tile_size': PUBLIC_MODEL_TILE_SIZE,
        'window_size': int(POSTPROCESSING['window_size']),
        'cells_and_nuclei': bool(public_network.cells_and_nuclei),
        'source_dataset': 'public InstanSeg release',
        'inference_device': PUBLIC_INFERENCE_DEVICE,
    }
    return public_runner, public_config


def _pad_for_model(image_tensor, multiple=INFERENCE_DIMENSION_MULTIPLE):
    original_shape = tuple(image_tensor.shape[-2:])
    pad_height = (-original_shape[0]) % int(multiple)
    pad_width = (-original_shape[1]) % int(multiple)
    if pad_height == 0 and pad_width == 0:
        return image_tensor, original_shape
    padding_mode = 'reflect'
    if pad_height >= original_shape[0] or pad_width >= original_shape[1]:
        padding_mode = 'replicate'
    padded = torch.nn.functional.pad(
        image_tensor,
        (0, pad_width, 0, pad_height),
        mode=padding_mode,
    )
    return padded, original_shape


def _crop_prediction_to_shape(prediction, original_shape):
    return prediction[..., :original_shape[0], :original_shape[1]]


_ORIGINAL_PUBLIC_FAST_SPARSE_IOU = loss_utils.fast_sparse_iou
_ORIGINAL_PUBLIC_FIND_CONNECTED_COMPONENTS = loss_utils.find_connected_components


def _safe_public_fast_sparse_iou(sparse_onehot):
    # The public implementation leaves COO duplicate indices uncoalesced and then
    # relies on an exact symmetry check after sparse matrix multiplication.
    sparse_onehot = sparse_onehot.coalesce()
    intersection = torch.sparse.mm(sparse_onehot, sparse_onehot.T).to_dense()
    sparse_sum = torch.sparse.sum(sparse_onehot, dim=(1,))[None].to_dense()
    union = sparse_sum.T + sparse_sum - intersection
    iou = intersection / union
    return (iou + iou.T) * 0.5


def _safe_public_find_connected_components(adjacency_matrix, max_iterations=100):
    symmetric = torch.logical_or(
        adjacency_matrix != 0,
        adjacency_matrix.T != 0,
    ).to(adjacency_matrix.dtype)
    return _ORIGINAL_PUBLIC_FIND_CONNECTED_COMPONENTS(symmetric, max_iterations)


@contextmanager
def _public_safe_postprocessing():
    previous_iou = loss_utils.fast_sparse_iou
    previous_components = loss_utils.find_connected_components
    loss_utils.fast_sparse_iou = _safe_public_fast_sparse_iou
    loss_utils.find_connected_components = _safe_public_find_connected_components
    try:
        yield
    finally:
        loss_utils.fast_sparse_iou = previous_iou
        loss_utils.find_connected_components = previous_components


class PublicPythonPredictor(nn.Module):
    def __init__(self, public_network):
        super().__init__()
        self.public_network = public_network
        public_device = str(next(iter(public_network.parameters())).device)
        self.postprocessor = LossInstanSeg(
            n_sigma=int(public_network.n_sigma),
            binary_loss_fn_str='lovasz_hinge',
            seed_loss_fn='binary_xloss',
            device=public_device,
            cells_and_nuclei=bool(public_network.cells_and_nuclei),
            window_size=int(public_network.default_window_size),
            dim_coords=int(public_network.dim_coords),
            dim_seeds=int(public_network.dim_seeds),
            feature_engineering_function='0',
        )
        # The public TorchScript graph uses this exact slow feature-engineering path.
        self.postprocessor.feature_engineering = loss_utils.feature_engineering_slow
        self.postprocessor.pixel_classifier = public_network.pixel_classifier

    def forward(self, image_batch, resolve_cell_and_nucleus=True, **kwargs):
        with torch.amp.autocast(image_batch.device.type, enabled=False):
            model_input = image_batch.clamp(min=-2, max=3)
            model_input, pad = _instanseg_padding(model_input, extra_pad=0)
            raw_output = self.public_network.fcn(model_input)
            raw_output = _recover_padding(raw_output, pad)
            with _public_safe_postprocessing():
                labels = [
                    self.postprocessor.postprocessing(
                        raw_sample,
                        device=image_batch.device,
                        classifier=self.public_network.pixel_classifier,
                        **kwargs,
                    )
                    for raw_sample in raw_output
                ]
            labels = torch.stack(labels)
            if resolve_cell_and_nucleus and labels.shape[1] == 2:
                # The public TorchScript model returns float labels before its
                # biological reconciliation; preserve that dtype for this helper.
                labels = loss_utils.resolve_cell_and_nucleus_boundaries(labels.float())
            return labels.float()


def predict_public_record(public_runner, image_tensor, config):
    public_kwargs = dict(POSTPROCESSING)
    model_input, original_shape = _pad_for_model(image_tensor)
    public_network = public_runner.instanseg
    public_predictor = PublicPythonPredictor(public_network)
    with torch.inference_mode():
        prediction = _sliding_window_inference(
            model_input,
            public_predictor,
            window_size=(int(FIXED_TILE_SIZE or config['tile_size']),) * 2,
            sw_device=config['inference_device'],
            device='cpu',
            batch_size=INFERENCE_BATCH_SIZE,
            output_channels=2,
            show_progress=False,
            instanseg_kwargs=public_kwargs,
        )
    prediction = prediction.squeeze(0).to(torch.int32)
    return _crop_prediction_to_shape(prediction, original_shape)


class RawTilePredictor(nn.Module):
    def __init__(self, model, postprocessor):
        super().__init__()
        self.model = model
        self.postprocessor = postprocessor

    def forward(self, image_batch, resolve_cell_and_nucleus=True, **kwargs):
        raw_output = self.model(image_batch)
        if isinstance(raw_output, (list, tuple)):
            raw_output = raw_output[0]
        return torch.stack([
            self.postprocessor(output, **kwargs)
            for output in raw_output
        ])


def predict_record(model, method, image_tensor, config):
    predictor = RawTilePredictor(model, method.postprocessing)
    tile_size = int(FIXED_TILE_SIZE or config['tile_size'])
    inference_kwargs = dict(POSTPROCESSING)
    model_input, original_shape = _pad_for_model(image_tensor)
    with torch.inference_mode():
        prediction = _sliding_window_inference(
            model_input,
            predictor,
            window_size=(tile_size, tile_size),
            sw_device=DEVICE,
            device='cpu',
            batch_size=INFERENCE_BATCH_SIZE,
            output_channels=2,
            show_progress=False,
            instanseg_kwargs=inference_kwargs,
        )
    prediction = prediction.squeeze(0).to(torch.int32)
    return _crop_prediction_to_shape(prediction, original_shape)

## Compute object-level metrics

Metrics are aggregated from instance matching at the same IoU thresholds used by the training code
(0.5 through 1.0). The primary score is a micro/object-count F1 at IoU 0.5, with the full threshold
curve retained. A nuclear row is generated only for records with a positive nuclear annotation; a cell
row is generated for all records with a positive cell annotation. Negative-label regions are treated as
unannotated and are removed from the prediction before matching.

In [ ]:
TARGET_NAMES = ('nuclei', 'cells')


def _evaluation_arrays(gt, prediction, target_index):
    gt_eval = gt[target_index].clone().to(torch.int32)
    prediction_eval = prediction[target_index].clone().to(torch.int32)

    valid_pixels = gt_eval >= 0
    if not bool(valid_pixels.all()):
        gt_eval = gt_eval.clone()
        prediction_eval = prediction_eval.clone()
        gt_eval[~valid_pixels] = 0
        prediction_eval[~valid_pixels] = 0

    if not bool((gt_eval > 0).any()):
        return None
    return gt_eval, prediction_eval


def _record_metric_rows(model_name, record_id, gt, prediction):
    rows = []
    for target_index, target_name in enumerate(TARGET_NAMES):
        arrays = _evaluation_arrays(gt, prediction, target_index)
        if arrays is None:
            continue
        gt_eval, prediction_eval = arrays
        stats = matching_torch(gt_eval, prediction_eval, THRESHOLDS)
        for threshold, stat in zip(THRESHOLDS, stats):
            rows.append({
                'model': model_name,
                'record_id': record_id,
                'target': target_name,
                'threshold': float(threshold),
                'tp': int(stat.tp),
                'fp': int(stat.fp),
                'fn': int(stat.fn),
                'precision': float(stat.precision),
                'recall': float(stat.recall),
                'f1': float(stat.f1),
                'n_true': int(stat.n_true),
                'n_pred': int(stat.n_pred),
                'ignored_pixels': int((gt[target_index] < 0).sum().item()),
            })
    return rows


def _aggregate_metric_rows(per_record_threshold):
    if per_record_threshold.empty:
        return pd.DataFrame(), pd.DataFrame()

    aggregate_rows = []
    for (model_name, target_name, threshold), group in per_record_threshold.groupby(
        ['model', 'target', 'threshold'],
        sort=True,
    ):
        tp = int(group['tp'].sum())
        fp = int(group['fp'].sum())
        fn = int(group['fn'].sum())
        precision = tp / max(tp + fp, 1e-10)
        recall = tp / max(tp + fn, 1e-10)
        f1 = 2 * tp / max(2 * tp + fp + fn, 1e-10)
        aggregate_rows.append({
            'model': model_name,
            'target': target_name,
            'threshold': float(threshold),
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'macro_f1': float(group['f1'].mean()),
            'n_true': int(group['n_true'].sum()),
            'n_pred': int(group['n_pred'].sum()),
            'n_records': int(group['record_id'].nunique()),
        })

    threshold_summary = pd.DataFrame(aggregate_rows)
    summary_rows = []
    for model_name in sorted(per_record_threshold['model'].unique()):
        model_thresholds = threshold_summary[threshold_summary['model'] == model_name]
        target_summaries = {}
        for target_name in TARGET_NAMES:
            target_rows = model_thresholds[model_thresholds['target'] == target_name]
            if target_rows.empty:
                continue
            at_05 = target_rows[target_rows['threshold'] == 0.5].iloc[0]
            target_summaries[target_name] = at_05
            summary_rows.append({
                'model': model_name,
                'target': target_name,
                'n_records': int(target_rows['n_records'].max()),
                'n_true': int(at_05['n_true']),
                'n_pred': int(at_05['n_pred']),
                'precision_iou50': float(at_05['precision']),
                'recall_iou50': float(at_05['recall']),
                'f1_iou50': float(at_05['f1']),
                'macro_f1_iou50': float(at_05['macro_f1']),
                'mean_f1_iou05_to_iou10': float(target_rows['f1'].mean()),
                'mean_macro_f1_iou05_to_iou10': float(target_rows['macro_f1'].mean()),
            })

        if target_summaries:
            mean_f1_curve = []
            mean_macro_curve = []
            for target_name in target_summaries:
                target_rows = model_thresholds[model_thresholds['target'] == target_name]
                mean_f1_curve.append(float(target_rows['f1'].mean()))
                mean_macro_curve.append(float(target_rows['macro_f1'].mean()))
            summary_rows.append({
                'model': model_name,
                'target': 'joint_mean_of_available_targets',
                'n_records': int(sum(row['n_records'] for row in target_summaries.values())),
                'n_true': int(sum(row['n_true'] for row in target_summaries.values())),
                'n_pred': int(sum(row['n_pred'] for row in target_summaries.values())),
                'precision_iou50': float(np.mean([
                    row['precision'] for row in target_summaries.values()
                ])),
                'recall_iou50': float(np.mean([
                    row['recall'] for row in target_summaries.values()
                ])),
                'f1_iou50': float(np.mean([
                    row['f1'] for row in target_summaries.values()
                ])),
                'macro_f1_iou50': float(np.mean([
                    row['macro_f1'] for row in target_summaries.values()
                ])),
                'mean_f1_iou05_to_iou10': float(np.mean(mean_f1_curve)),
                'mean_macro_f1_iou05_to_iou10': float(np.mean(mean_macro_curve)),
            })

    return pd.DataFrame(summary_rows), threshold_summary


def build_metric_tables(model_name, prediction_pairs):
    per_record_rows = []
    for pair in prediction_pairs:
        per_record_rows.extend(_record_metric_rows(
            model_name,
            pair['record_id'],
            pair['ground_truth'],
            pair['prediction'],
        ))
    per_record_threshold = pd.DataFrame(per_record_rows)
    summary, threshold_summary = _aggregate_metric_rows(per_record_threshold)
    return summary, threshold_summary, per_record_threshold

In [ ]:
SUMMARY_DF = pd.DataFrame()
THRESHOLD_DF = pd.DataFrame()
PER_RECORD_DF = pd.DataFrame()
GALLERY_PREDICTIONS = {}
GALLERY_INPUTS = {}

gallery_candidates = [
    record['record_id'] for record in MANIFEST_RECORDS
    if record['nucleus_annotation'] and record['cell_annotation']
]
if len(gallery_candidates) < GALLERY_COUNT:
    gallery_candidates = [record['record_id'] for record in MANIFEST_RECORDS]
GALLERY_RECORD_IDS = gallery_candidates[:GALLERY_COUNT]

if RUN_EVALUATION:
    if not EVALUATION_MODEL_NAMES:
        raise RuntimeError('No selected models.')
    if DEVICE == 'cpu':
        print('Running on CPU; this is useful for a small smoke test but will be slow.')
    summary_parts = []
    threshold_parts = []
    per_record_parts = []

    for model_number, model_name in enumerate(EVALUATION_MODEL_NAMES, start=1):
        print(f'[{model_number}/{len(EVALUATION_MODEL_NAMES)}] Loading {model_name}')
        started = time.perf_counter()
        public_runner = None
        if model_name == PUBLIC_MODEL_LABEL:
            public_runner, config = load_public_model_for_evaluation()
            model = public_runner.instanseg
            method = None
        else:
            model, config, method = load_checkpoint_for_evaluation(model_name)
        prediction_pairs = []

        for record in MANIFEST_RECORDS:
            record_id = record['record_id']
            source_index, item = RECORD_LOOKUP[record_id]
            image_tensor, ground_truth = get_prepared_record(
                source_index,
                item,
                config['requested_pixel_size'],
            )
            if public_runner is None:
                prediction = predict_record(model, method, image_tensor, config).cpu()
            else:
                prediction = predict_public_record(public_runner, image_tensor, config).cpu()
            prediction_pairs.append({
                'record_id': record_id,
                'ground_truth': ground_truth.cpu(),
                'prediction': prediction,
            })
            if record_id in GALLERY_RECORD_IDS:
                GALLERY_INPUTS[(model_name, record_id)] = (
                    image_tensor.cpu(),
                    ground_truth.cpu(),
                )
                GALLERY_PREDICTIONS[(model_name, record_id)] = prediction

        model_summary, model_thresholds, model_per_record = build_metric_tables(
            model_name,
            prediction_pairs,
        )
        summary_parts.append(model_summary)
        threshold_parts.append(model_thresholds)
        per_record_parts.append(model_per_record)

        elapsed = time.perf_counter() - started
        print(f'Completed {model_name} in {elapsed / 60:.1f} minutes.')

        del prediction_pairs
        del model
        del method
        if public_runner is not None:
            del public_runner
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    SUMMARY_DF = pd.concat(summary_parts, ignore_index=True)
    THRESHOLD_DF = pd.concat(threshold_parts, ignore_index=True)
    PER_RECORD_DF = pd.concat(per_record_parts, ignore_index=True)

    if SAVE_ARTIFACTS:
        RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
        checkpoint_tag = Path(CHECKPOINT_FILENAME).stem
        SUMMARY_DF.to_csv(
            RESULTS_ROOT / f'cpdmi_validation_summary_{checkpoint_tag}.csv',
            index=False,
        )
        THRESHOLD_DF.to_csv(
            RESULTS_ROOT / f'cpdmi_validation_thresholds_{checkpoint_tag}.csv',
            index=False,
        )
        PER_RECORD_DF.to_csv(
            RESULTS_ROOT / f'cpdmi_validation_per_record_{checkpoint_tag}.csv',
            index=False,
        )

        provenance = {
            'created_utc': datetime.now(timezone.utc).isoformat(),
            'dataset_file': str(DATASET_FILE),
            'manifest_file': str(MANIFEST_PATH),
            'instanseg_source': str(instanseg_import_path),
            'torch_version': torch.__version__,
            'device': DEVICE,
            'checkpoint_filename': CHECKPOINT_FILENAME,
            'postprocessing': POSTPROCESSING,
            'thresholds': THRESHOLDS,
            'fixed_tile_size': FIXED_TILE_SIZE,
            'inference_batch_size': INFERENCE_BATCH_SIZE,
            'models': MODEL_ROWS,
            'evaluation_model_names': EVALUATION_MODEL_NAMES,
            'public_model': {
                'name': PUBLIC_MODEL_NAME,
                'version': PUBLIC_MODEL_VERSION,
                'cache_root': str(PUBLIC_MODEL_CACHE_ROOT),
                'tile_size': PUBLIC_MODEL_TILE_SIZE,
                'inference_device': PUBLIC_INFERENCE_DEVICE,
            },
        }
        (RESULTS_ROOT / f'cpdmi_validation_provenance_{checkpoint_tag}.json').write_text(
            json.dumps(provenance, indent=2, default=str) + '\n'
        )
        print('Saved comparison artifacts under', RESULTS_ROOT)
else:
    print('Evaluation is gated. Set RUN_EVALUATION=True and rerun this cell on a CUDA allocation.')

## Review the common-validation results

The joint row is the arithmetic mean of the available nuclear and cell rows. It is useful as a compact
summary, while the compartment rows and the saved per-record table should drive model selection. Because
the CPDMI validation set contains many cell-only records, cell and joint values should not be interpreted
as if they were based on equal numbers of annotated compartments.

In [ ]:
if SUMMARY_DF.empty:
    print('No evaluation results in memory. Run the gated evaluation cell first.')
else:
    columns = [
        'model', 'target', 'n_records', 'n_true', 'n_pred',
        'precision_iou50', 'recall_iou50', 'f1_iou50',
        'macro_f1_iou50', 'mean_f1_iou05_to_iou10',
    ]
    display(
        SUMMARY_DF[columns]
        .sort_values(['target', 'f1_iou50'], ascending=[True, False])
        .reset_index(drop=True)
    )

In [ ]:
if THRESHOLD_DF.empty:
    print('No threshold results in memory.')
else:
    figure, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for axis, target_name in zip(axes, TARGET_NAMES):
        target_data = THRESHOLD_DF[THRESHOLD_DF['target'] == target_name]
        for model_name in EVALUATION_MODEL_NAMES:
            curve = target_data[target_data['model'] == model_name]
            if not curve.empty:
                axis.plot(
                    curve['threshold'],
                    curve['f1'],
                    marker='o',
                    linewidth=1.5,
                    label=model_name,
                )
        axis.set_title(f'{target_name.title()} — fixed CPDMI validation')
        axis.set_xlabel('IoU threshold')
        axis.set_ylabel('micro/object-count F1')
        axis.set_ylim(0, 1)
        axis.grid(alpha=0.25)
    axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    figure.tight_layout()
    plt.show()

In [ ]:
if PER_RECORD_DF.empty:
    print('No per-record results in memory.')
else:
    per_record_iou50 = PER_RECORD_DF[PER_RECORD_DF['threshold'] == 0.5].copy()
    per_record_iou50 = per_record_iou50.merge(
        MANIFEST_TABLE[['record_id', 'filename', 'platform', 'nucleus_annotation', 'cell_annotation']],
        on='record_id',
        how='left',
    )
    per_record_iou50 = (
        per_record_iou50
        .sort_values(['target', 'record_id', 'model'])
        .reset_index(drop=True)
    )
    display(
        per_record_iou50[[
            'record_id', 'filename', 'platform', 'target', 'model', 'f1',
            'precision', 'recall', 'n_true', 'n_pred',
        ]]
    )

## Optional qualitative gallery

The gallery uses the first two records with both compartments annotated (or the first two records if
that subset is empty). Source images are shown after the model-specific physical rescaling, while
ground-truth and prediction boundaries use the exact same label rasters scored above.

In [ ]:
from skimage.segmentation import find_boundaries


def _display_base(image_tensor):
    image = image_tensor.numpy()
    base = image.mean(axis=0)
    low, high = np.percentile(base, (1, 99))
    return np.clip((base - low) / max(float(high - low), 1e-6), 0, 1)



def _boundary_overlay(base, labels):
    labels = np.asarray(labels).copy()
    labels[labels < 0] = 0
    rgb = np.repeat(base[..., None], 3, axis=-1)
    cell_boundary = find_boundaries(labels[1], mode='outer')
    nucleus_boundary = find_boundaries(labels[0], mode='outer')
    rgb[cell_boundary] = (1.0, 0.2, 0.1)
    rgb[nucleus_boundary] = (0.1, 0.8, 1.0)
    return rgb


if not GALLERY_PREDICTIONS:
    print('No gallery predictions in memory. Run the gated evaluation cell first.')
else:
    for record_id in GALLERY_RECORD_IDS:
        available_models = [
            model_name for model_name in EVALUATION_MODEL_NAMES
            if (model_name, record_id) in GALLERY_PREDICTIONS
        ]
        figure, axes = plt.subplots(
            len(available_models),
            3,
            figsize=(12, max(3, 3 * len(available_models))),
            squeeze=False,
        )
        for row_index, model_name in enumerate(available_models):
            image_tensor, ground_truth = GALLERY_INPUTS[(model_name, record_id)]
            prediction = GALLERY_PREDICTIONS[(model_name, record_id)]
            base = _display_base(image_tensor)
            axes[row_index, 0].imshow(base, cmap='gray')
            axes[row_index, 0].set_title(f'{model_name}\nsource')
            axes[row_index, 1].imshow(_boundary_overlay(base, ground_truth))
            axes[row_index, 1].set_title('ground truth')
            axes[row_index, 2].imshow(_boundary_overlay(base, prediction))
            axes[row_index, 2].set_title('prediction')
            for axis in axes[row_index]:
                axis.axis('off')
        figure.suptitle(record_id, y=1.02)
        figure.tight_layout()
        plt.show()

## Interpretation checklist

- Use the manifest file as the comparison contract. If the dataset is regenerated, create a new results
  directory only after reviewing the changed hashes.
- Compare the CPDMI-only and mixed rows directly because they share the same CPDMI record IDs here.
  The mixed models' training-time pooled F1 is not a substitute for this result.
- Check `n_records` before comparing compartments. Targets with no positive ground-truth instances are
  omitted by the metric code; in the current manifest, nuclei are scored on 7 records / 3,377 objects,
  while cells are scored on 30 records / 14,619 objects.
- `f1_iou50` is the pooled object-level F1 from summed TP/FP/FN. `macro_f1_iou50` is the mean of the
  per-record F1 values. The joint row is only an arithmetic mean of the available compartment rows.
- Near-ties can change rank with aggregation: at IoU 0.50, pooled cell F1 is 0.7291 for the public
  model versus 0.7314 for the best mixed model, while macro cell F1 is 0.7342 versus 0.7335.
- Treat the public fluorescence_nuclei_and_cells v0.1.1 row as the pretrained baseline. It uses the
  public wrapper's raw network with a Python-equivalent postprocessor (the public TorchScript postprocessor
  is avoided because it can fail on CUDA sparse-symmetry checks and can abort on CPU); trained checkpoints
  use the explicit training postprocessing dictionary.
- Treat the 0.5 versus 0.325 rows as a resolution comparison, not as a pure architecture comparison:
  each model is evaluated on its own trained physical grid. The underlying records are identical, but
  the prepared images and nearest-neighbor label rasters are not pixel-for-pixel identical across scales.
  Pixel-valued postprocessing settings also have different physical meanings at 0.325 versus 0.5 µm/pixel.
- Keep the native-grid score as the operational result for downstream use. For an apples-to-apples model
  comparison, add a second report that label-aware resamples every prediction and ground truth to one
  common physical grid before matching; this separates model quality from output-grid/discretization effects.
- The default postprocessing matches the trainer's explicit 0.53/4/0.5/0.5/128/10 settings. If you
  tune thresholds, create a separate results directory or checkpoint tag and preserve the settings in
  the provenance JSON.
- This notebook is analysis-only. It does not alter model checkpoints, the saved dataset, or pipeline
  artifacts.